# k-means4

샘플 좌표에서 `k`별 silhouette score를 비교한 뒤, 가장 좋은 `k`로 전체 체크인에 대해 `MiniBatchKMeans`를 학습합니다.

In [2]:
from datetime import datetime, timezone

import numpy as np
from pymongo import MongoClient, UpdateOne
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from tqdm.auto import tqdm


In [3]:
HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

CHECKIN_COL = "2. US_checkin_v2"
FREQ_COL = "3. user_cluster_frequency_bestk"
META_COL = "3. loc_kmeans_bestk_meta"
CENTROID_COL = "3. loc_cluster_centroids_bestk"
ASSIGN_FIELD = "loc_cluster_id_bestk"

K_MIN = 2
K_MAX = 50
SIL_SAMPLE = 20000
TRAIN_BATCH = 20000
ASSIGN_BATCH = 20000
RANDOM_STATE = 42

client = MongoClient(host=HOST, port=PORT)
db = client[DB_NAME]
checkin_col = db[CHECKIN_COL]


query = {"latitude": {"$ne": None}, "longitude": {"$ne": None}}
n_points = checkin_col.count_documents(query)

print("checkins with coordinates:", n_points)
print("assignment field:", ASSIGN_FIELD)
print("frequency output:", FREQ_COL)
print("centroid output:", CENTROID_COL)


checkins with coordinates: 897701
assignment field: loc_cluster_id_bestk
frequency output: 3. user_cluster_frequency_bestk


## 1. silhouette 계산용 샘플 좌표 로드

In [4]:
if n_points < 2:
    raise RuntimeError(f"클러스터링할 좌표가 부족합니다. n_points={n_points}")

sample_size = min(SIL_SAMPLE, n_points)
cursor = checkin_col.aggregate([
    {"$match": query},
    {"$sample": {"size": sample_size}},
    {"$project": {"_id": 0, "latitude": 1, "longitude": 1}},
], allowDiskUse=True)

sample_xy = np.array(
    [[doc["latitude"], doc["longitude"]] for doc in cursor],
    dtype=np.float32,
)

print("sample shape:", sample_xy.shape)


sample shape: (20000, 2)


## 2. k 후보별 silhouette score 비교

In [5]:
k_max = min(K_MAX, len(sample_xy) - 1)
if k_max < K_MIN:
    raise RuntimeError(f"실루엣 계산 불가: sample 수가 너무 적습니다. sample={len(sample_xy)}")

silhouette_results = []
best_k = None
best_score = -1.0

for k in range(K_MIN, k_max + 1):
    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=min(TRAIN_BATCH, len(sample_xy)),
        n_init="auto",
        max_iter=200,
        reassignment_ratio=0.01,
    )
    labels = model.fit_predict(sample_xy)

    if np.unique(labels).shape[0] < 2:
        print(f"skip k={k}: 하나의 클러스터만 생성되었습니다.")
        continue

    score = silhouette_score(sample_xy, labels, metric="euclidean")
    silhouette_results.append({"k": int(k), "silhouette": float(score)})
    print(f"k={k:>3} | silhouette={score:.4f}")

    if score > best_score:
        best_k = k
        best_score = float(score)

if best_k is None:
    raise RuntimeError("최적 k를 찾지 못했습니다.")

print(f"\nbest_k={best_k}, best_silhouette={best_score:.4f}")
silhouette_results


k=  2 | silhouette=0.4266
k=  3 | silhouette=0.4464
k=  4 | silhouette=0.3848
k=  5 | silhouette=0.5548
k=  6 | silhouette=0.5108
k=  7 | silhouette=0.2297
k=  8 | silhouette=0.2454
k=  9 | silhouette=0.2272
k= 10 | silhouette=0.2345
k= 11 | silhouette=0.1618
k= 12 | silhouette=0.2180
k= 13 | silhouette=0.1676
k= 14 | silhouette=0.1942
k= 15 | silhouette=0.2869
k= 16 | silhouette=0.1881
k= 17 | silhouette=0.1232
k= 18 | silhouette=0.0694
k= 19 | silhouette=0.1713
k= 20 | silhouette=0.1638
k= 21 | silhouette=0.0286
k= 22 | silhouette=0.0795
k= 23 | silhouette=0.0391
k= 24 | silhouette=0.0930
k= 25 | silhouette=0.0918
k= 26 | silhouette=0.0992
k= 27 | silhouette=0.1739
k= 28 | silhouette=0.0723
k= 29 | silhouette=0.0540
k= 30 | silhouette=0.0510
k= 31 | silhouette=0.0715
k= 32 | silhouette=0.0956
k= 33 | silhouette=0.0694
k= 34 | silhouette=0.0656
k= 35 | silhouette=0.0158
k= 36 | silhouette=0.0549
k= 37 | silhouette=0.0768
k= 38 | silhouette=0.0428
k= 39 | silhouette=0.1474
k= 40 | silh

[{'k': 2, 'silhouette': 0.42657288908958435},
 {'k': 3, 'silhouette': 0.44635728001594543},
 {'k': 4, 'silhouette': 0.3848457932472229},
 {'k': 5, 'silhouette': 0.5548425316810608},
 {'k': 6, 'silhouette': 0.5107842087745667},
 {'k': 7, 'silhouette': 0.22967353463172913},
 {'k': 8, 'silhouette': 0.2453737109899521},
 {'k': 9, 'silhouette': 0.22723951935768127},
 {'k': 10, 'silhouette': 0.2344641387462616},
 {'k': 11, 'silhouette': 0.1618427187204361},
 {'k': 12, 'silhouette': 0.21798095107078552},
 {'k': 13, 'silhouette': 0.16763441264629364},
 {'k': 14, 'silhouette': 0.1942317634820938},
 {'k': 15, 'silhouette': 0.2868565320968628},
 {'k': 16, 'silhouette': 0.18809816241264343},
 {'k': 17, 'silhouette': 0.12315571308135986},
 {'k': 18, 'silhouette': 0.06935791671276093},
 {'k': 19, 'silhouette': 0.17133016884326935},
 {'k': 20, 'silhouette': 0.16380971670150757},
 {'k': 21, 'silhouette': 0.028586266562342644},
 {'k': 22, 'silhouette': 0.07948973029851913},
 {'k': 23, 'silhouette': 0.0

## 3. best_k로 전체 체크인 MiniBatchKMeans 학습

In [6]:
best_model = MiniBatchKMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    batch_size=TRAIN_BATCH,
    n_init="auto",
    max_iter=200,
    reassignment_ratio=0.01,
)

cursor = checkin_col.find(query, {"_id": 0, "latitude": 1, "longitude": 1}, no_cursor_timeout=True)
buf = []

pbar = tqdm(total=n_points, desc=f"Training best_k={best_k}")
for doc in cursor:
    buf.append([doc["latitude"], doc["longitude"]])
    if len(buf) >= TRAIN_BATCH:
        best_model.partial_fit(np.asarray(buf, dtype=np.float32))
        pbar.update(len(buf))
        buf.clear()

if buf:
    best_model.partial_fit(np.asarray(buf, dtype=np.float32))
    pbar.update(len(buf))

pbar.close()
cursor.close()
print("trained best model with k =", best_model.n_clusters)


/home/gpuadmin/anaconda3/lib/python3.12/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Training best_k=5:   0%|          | 0/897701 [00:00<?, ?it/s]

trained best model with k = 5


## 4. 전체 체크인에 최종 cluster id 부여

In [7]:
cursor = checkin_col.find(
    query,
    {"_id": 1, "latitude": 1, "longitude": 1},
    no_cursor_timeout=True,
)

ops = []
buf_ids = []
buf_xy = []

pbar = tqdm(total=n_points, desc=f"Assigning {ASSIGN_FIELD}")
for doc in cursor:
    buf_ids.append(doc["_id"])
    buf_xy.append([doc["latitude"], doc["longitude"]])

    if len(buf_ids) >= ASSIGN_BATCH:
        labels = best_model.predict(np.asarray(buf_xy, dtype=np.float32))
        for _id, label in zip(buf_ids, labels):
            ops.append(UpdateOne({"_id": _id}, {"$set": {ASSIGN_FIELD: int(label)}}))

        checkin_col.bulk_write(ops, ordered=False)
        pbar.update(len(buf_ids))
        ops.clear()
        buf_ids.clear()
        buf_xy.clear()

if buf_ids:
    labels = best_model.predict(np.asarray(buf_xy, dtype=np.float32))
    for _id, label in zip(buf_ids, labels):
        ops.append(UpdateOne({"_id": _id}, {"$set": {ASSIGN_FIELD: int(label)}}))
    checkin_col.bulk_write(ops, ordered=False)
    pbar.update(len(buf_ids))

pbar.close()
cursor.close()
print("done:", ASSIGN_FIELD, "updated")


/home/gpuadmin/anaconda3/lib/python3.12/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Assigning loc_cluster_id_bestk:   0%|          | 0/897701 [00:00<?, ?it/s]

done: loc_cluster_id_bestk updated


## 5. user_id별 cluster 방문 빈도 저장

In [10]:
freq_col = db[FREQ_COL]
meta_col = db[META_COL]
centroid_col = db[CENTROID_COL]

now = datetime.now(timezone.utc)
# MongoDB의 _id 인덱스는 기본적으로 unique 이므로 user_id만 인덱싱합니다.
freq_col.create_index("user_id", unique=True)

freq_col.create_index("user_id", unique=True)

pipeline = [
    {"$match": {ASSIGN_FIELD: {"$exists": True}}},
    {
        "$group": {
            "_id": {"user_id": "$user_id", "cluster_id": f"${ASSIGN_FIELD}"},
            "count": {"$sum": 1},
        }
    },
    {
        "$group": {
            "_id": "$_id.user_id",
            "pairs": {"$push": {"k": {"$toString": "$_id.cluster_id"}, "v": "$count"}},
            "total_checkins": {"$sum": "$count"},
        }
    },
    {
        "$addFields": {
            "freq": {"$arrayToObject": "$pairs"},
            "user_id": "$_id",
            "k": int(best_k),
            "silhouette": float(best_score),
            "model": "MiniBatchKMeans",
            "assignment_field": ASSIGN_FIELD,
            "updated_at": now,
        }
    },
    {"$project": {"pairs": 0}},
    {
        "$merge": {
            "into": FREQ_COL,
            "on": "_id",
            "whenMatched": "replace",
            "whenNotMatched": "insert",
        }
    },
]

list(checkin_col.aggregate(pipeline, allowDiskUse=True))
print("saved:", FREQ_COL)


saved: 3. user_cluster_frequency_bestk


## 6. 클러스터 중심 좌표 및 실행 메타데이터 저장

In [11]:
centroid_docs = []
for cluster_id, (latitude, longitude) in enumerate(best_model.cluster_centers_):
    centroid_docs.append({
        "_id": f"best_k_{best_k}_cluster_{cluster_id}",
        "cluster_id": int(cluster_id),
        "k": int(best_k),
        "latitude": float(latitude),
        "longitude": float(longitude),
        "assign_field": ASSIGN_FIELD,
        "source_collection": CHECKIN_COL,
        "updated_at": datetime.now(timezone.utc),
    })

for doc in centroid_docs:
    centroid_col.replace_one({"_id": doc["_id"]}, doc, upsert=True)

print("saved centroids:", CENTROID_COL, len(centroid_docs))

meta_doc = {
    "_id": f"best_k_{best_k}",
    "k": int(best_k),
    "silhouette": float(best_score),
    "n_points": int(n_points),
    "sample_size": int(len(sample_xy)),
    "assign_field": ASSIGN_FIELD,
    "freq_collection": FREQ_COL,
    "centroids": best_model.cluster_centers_.tolist(),
    "silhouette_results": silhouette_results,
    "updated_at": datetime.now(timezone.utc),
}

meta_doc["centroid_collection"] = CENTROID_COL

meta_col.replace_one({"_id": meta_doc["_id"]}, meta_doc, upsert=True)
print("saved meta:", META_COL)
meta_doc


saved meta: 3. loc_kmeans_bestk_meta


{'_id': 'best_k_5',
 'k': 5,
 'silhouette': 0.5548425316810608,
 'n_points': 897701,
 'sample_size': 20000,
 'assign_field': 'loc_cluster_id_bestk',
 'freq_collection': '3. user_cluster_frequency_bestk',
 'centroids': [[40.82884979248047, -73.99425506591797],
  [40.825687408447266, -73.99960327148438],
  [40.654666900634766, -74.00653076171875],
  [40.772438049316406, -73.8897933959961],
  [40.660621643066406, -73.78041076660156]],
 'silhouette_results': [{'k': 2, 'silhouette': 0.42657288908958435},
  {'k': 3, 'silhouette': 0.44635728001594543},
  {'k': 4, 'silhouette': 0.3848457932472229},
  {'k': 5, 'silhouette': 0.5548425316810608},
  {'k': 6, 'silhouette': 0.5107842087745667},
  {'k': 7, 'silhouette': 0.22967353463172913},
  {'k': 8, 'silhouette': 0.2453737109899521},
  {'k': 9, 'silhouette': 0.22723951935768127},
  {'k': 10, 'silhouette': 0.2344641387462616},
  {'k': 11, 'silhouette': 0.1618427187204361},
  {'k': 12, 'silhouette': 0.21798095107078552},
  {'k': 13, 'silhouette': 0.

## 7. k=5로 클러스터링 및 시각화

In [6]:
import pandas as pd
import plotly.express as px

k_fixed = 5

model_fixed = MiniBatchKMeans(
    n_clusters=k_fixed,
    random_state=RANDOM_STATE,
    batch_size=TRAIN_BATCH,
    n_init="auto",
    max_iter=200,
    reassignment_ratio=0.01,
)

cursor = checkin_col.find(query, {"_id": 0, "latitude": 1, "longitude": 1}, no_cursor_timeout=True)
buf = []

for doc in cursor:
    buf.append([doc["latitude"], doc["longitude"]])
    if len(buf) >= TRAIN_BATCH:
        model_fixed.partial_fit(np.asarray(buf, dtype=np.float32))
        buf.clear()

if buf:
    model_fixed.partial_fit(np.asarray(buf, dtype=np.float32))

cursor.close()
print("trained fixed model with k =", model_fixed.n_clusters)

# 샘플 데이터에 predict
labels_fixed = model_fixed.predict(sample_xy)

# 시각화
df = pd.DataFrame({
    'latitude': sample_xy[:, 0],
    'longitude': sample_xy[:, 1],
    'cluster': labels_fixed
})

fig = px.scatter_geo(df, lat='latitude', lon='longitude', color='cluster', scope='usa', title=f'K-means Clustering on US Map with k={k_fixed}')
fig.show()

/home/gpuadmin/.local/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


trained fixed model with k = 5
